In [60]:
import pandas as pd
import numpy as np

In [61]:
df = pd.read_csv('../Data/healthcare_data.csv')
df.drop(df.columns[0], axis=1, inplace=True)
df.head(5)

,patient_id,gender,age,has_hypertension,has_heart_disease,marital_status,employment_type,residence,glucose_level,bmi_value,smoking_habit,stroke_event,age_group,risk_score,high_glucose,bmi_category,lifestyle_risk
0,30669,M,3.0,0,0,0,other,Rural,95.12,18.0,unknown,0,young,0,0,underweight,low
1,30468,M,58.0,1,0,1,working,Urban,87.96,39.2,non_smoker,0,middle,1,0,obese,low
2,16523,F,8.0,0,0,0,working,Urban,110.89,17.6,unknown,0,young,0,0,underweight,low
3,56543,F,70.0,0,0,1,working,Rural,69.04,35.9,ex_smoker,0,senior,0,0,obese,medium
4,46136,M,14.0,0,0,0,other,Rural,161.28,19.1,unknown,0,young,0,1,normal,low


In [62]:
df.patient_id.nunique()

5110

In [63]:
#Check for Exact Duplicates (Data Entry Errors)
exact_duplicates = df.duplicated().sum()
print(f"Exact row duplicates (every column matches): {exact_duplicates}")

# Check for Multiple Hospital Visits
# This checks if the ID is the same, but the medical data changed
id_duplicates = df.duplicated(subset=['patient_id']).sum()
print(f"Total ID duplicates: {id_duplicates}")
print(f"Patients with multiple DIFFERENT records: {id_duplicates - exact_duplicates}")

Exact row duplicates (every column matches): 4612
Total ID duplicates: 4612
Patients with multiple DIFFERENT records: 0


In [64]:
df = df.drop_duplicates()
print(f"Clean dataset shape: {df.shape}")

Clean dataset shape: (5110, 17)


In [65]:
# Check the class balance
print(df['stroke_event'].value_counts(normalize=True) * 100)

stroke_event
0    95.127202
1     4.872798
Name: proportion, dtype: float64


In [66]:
df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
patient_id,5110.0,36517.829354,21161.721625,67.00,17741.250,36932.000,54682.00,72940.00
age,5110.0,43.226614,22.612647,0.08,25.000,45.000,61.00,82.00
has_hypertension,5110.0,0.097456,0.296607,0.00,0.000,0.000,0.00,1.00
has_heart_disease,5110.0,0.054012,0.226063,0.00,0.000,0.000,0.00,1.00
marital_status,5110.0,0.656164,0.475034,0.00,0.000,1.000,1.00,1.00
glucose_level,5110.0,106.147677,45.283560,55.12,77.245,91.885,114.09,271.74
bmi_value,4909.0,28.893237,7.854067,10.30,23.500,28.100,33.10,97.60
stroke_event,5110.0,0.048728,0.215320,0.00,0.000,0.000,0.00,1.00
risk_score,5110.0,0.151468,0.391924,0.00,0.000,0.000,0.00,2.00
high_glucose,5110.0,0.160665,0.367258,0.00,0.000,0.000,0.00,1.00


In [67]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   patient_id         5110 non-null   int64  
 1   gender             5110 non-null   str    
 2   age                5110 non-null   float64
 3   has_hypertension   5110 non-null   int64  
 4   has_heart_disease  5110 non-null   int64  
 5   marital_status     5110 non-null   int64  
 6   employment_type    5110 non-null   str    
 7   residence          5110 non-null   str    
 8   glucose_level      5110 non-null   float64
 9   bmi_value          4909 non-null   float64
 10  smoking_habit      5110 non-null   str    
 11  stroke_event       5110 non-null   int64  
 12  age_group          5110 non-null   str    
 13  risk_score         5110 non-null   int64  
 14  high_glucose       5110 non-null   int64  
 15  bmi_category       5110 non-null   str    
 16  lifestyle_risk     5110 non-null   

In [58]:
# Create a boolean mask for missing BMI
df['bmi_missing'] = df['bmi_value'].isnull()

# Check distribution of the target variable for missing vs. non-missing BMI
print(df.groupby('bmi_missing')['stroke_event'].mean())

# Check against demographics (e.g., are we missing BMI mostly for older patients?)
print(df.groupby('bmi_missing')['age'].mean())

bmi_missing
False    0.042575
True     0.199005
Name: stroke_event, dtype: float64
bmi_missing
False    42.865374
True     52.049154
Name: age, dtype: float64


In [69]:
df['bmi_was_missing'] = df['bmi_value'].isnull().astype(int)
df.head(5)

,patient_id,gender,age,has_hypertension,has_heart_disease,marital_status,employment_type,residence,glucose_level,bmi_value,smoking_habit,stroke_event,age_group,risk_score,high_glucose,bmi_category,lifestyle_risk,bmi_was_missing
0,30669,M,3.0,0,0,0,other,Rural,95.12,18.0,unknown,0,young,0,0,underweight,low,0
1,30468,M,58.0,1,0,1,working,Urban,87.96,39.2,non_smoker,0,middle,1,0,obese,low,0
2,16523,F,8.0,0,0,0,working,Urban,110.89,17.6,unknown,0,young,0,0,underweight,low,0
3,56543,F,70.0,0,0,1,working,Rural,69.04,35.9,ex_smoker,0,senior,0,0,obese,medium,0
4,46136,M,14.0,0,0,0,other,Rural,161.28,19.1,unknown,0,young,0,1,normal,low,0


In [70]:
# group the patients by their existing 'bmi_category' and 'gender'.
# fill the blank BMI with the median of that specific group.

df['bmi_value'] = df.groupby(['bmi_category', 'gender'])['bmi_value'].transform(
    lambda x: x.fillna(x.median())
)

In [71]:
print(df[df['bmi_was_missing'] == 1][['gender', 'bmi_category', 'bmi_value', 'bmi_was_missing']].head())

    gender bmi_category  bmi_value  bmi_was_missing
93       M        obese       33.8                1
111      F        obese       35.4                1
183      F        obese       35.4                1
228      M        obese       33.8                1
230      M        obese       33.8                1


In [72]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   patient_id         5110 non-null   int64  
 1   gender             5110 non-null   str    
 2   age                5110 non-null   float64
 3   has_hypertension   5110 non-null   int64  
 4   has_heart_disease  5110 non-null   int64  
 5   marital_status     5110 non-null   int64  
 6   employment_type    5110 non-null   str    
 7   residence          5110 non-null   str    
 8   glucose_level      5110 non-null   float64
 9   bmi_value          5110 non-null   float64
 10  smoking_habit      5110 non-null   str    
 11  stroke_event       5110 non-null   int64  
 12  age_group          5110 non-null   str    
 13  risk_score         5110 non-null   int64  
 14  high_glucose       5110 non-null   int64  
 15  bmi_category       5110 non-null   str    
 16  lifestyle_risk     5110 non-null   

In [73]:
print(df.gender.value_counts())
print('===============================================================')
print(df.has_hypertension.value_counts())
print('===============================================================')
print(df.has_heart_disease.value_counts())
print('===============================================================')
print(df.marital_status.value_counts())
print('===============================================================')
print(df.employment_type.value_counts())
print('===============================================================')
print(df.residence.value_counts())
print('===============================================================')
print(df.smoking_habit.value_counts())
print('===============================================================')
print(df.stroke_event.value_counts())
print('===============================================================')
print(df.age_group.value_counts())
print('===============================================================')
print(df.high_glucose.value_counts())
print('===============================================================')
print(df.bmi_category.value_counts())
print('===============================================================')
print(df.lifestyle_risk.value_counts())

gender
F    2995
M    2115
Name: count, dtype: int64
has_hypertension
0    4612
1     498
Name: count, dtype: int64
has_heart_disease
0    4834
1     276
Name: count, dtype: int64
marital_status
1    3353
0    1757
Name: count, dtype: int64
employment_type
working       3744
other          709
government     657
Name: count, dtype: int64
residence
Urban    2596
Rural    2514
Name: count, dtype: int64
smoking_habit
non_smoker        1892
unknown           1544
ex_smoker          885
current_smoker     789
Name: count, dtype: int64
stroke_event
0    4861
1     249
Name: count, dtype: int64
age_group
middle    2219
young     1515
senior    1376
Name: count, dtype: int64
high_glucose
0    4289
1     821
Name: count, dtype: int64
bmi_category
obese          2121
overweight     1409
normal         1243
underweight     337
Name: count, dtype: int64
lifestyle_risk
low       3840
medium     885
high       385
Name: count, dtype: int64


In [74]:
df.to_csv('../Data/healthcare_data_cleaned.csv', index=False)